In [0]:
dbutils.widgets.text("p_file_date","2021-03-21")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS f1_presentation.calculated_race_results
          (
              race_year INT,
              team_name STRING,
              driver_id INT,
              driver_name STRING,
              race_id INT,
              position INT,
              points INT,
              calculated_points INT,
              created_date TIMESTAMP,
              updated_date TIMESTAMP
          )
          USING DELTA
          """)

In [0]:
%sql
SHOW TABLES IN f1_presentation

In [0]:

spark.sql(f"""CREATE OR REPLACE TEMP VIEW race_results_updated
AS
SELECT 
rc.race_year,
c.name AS team_name, 
d.driver_id,
d.name AS driver_name, 
rc.race_id,
r.position,
r.points,
11 - position AS calculated_points
FROM f1_processed.results AS r
JOIN f1_processed.drivers AS d ON (d.driver_id = r.driver_id)
JOIN f1_processed.constructors AS c on (c.constructor_id = r.constructor_id)
JOIN f1_processed.races AS rc on (rc.race_id = r.race_id)
WHERE r.position <= 10 AND r.file_date = '{v_file_date}'""")

In [0]:
%sql
SELECT COUNT(*) FROM race_results_updated

In [0]:

spark.sql(f"""MERGE INTO f1_presentation.calculated_race_results tgt
USING race_results_updated upd
ON (tgt.driver_id = upd.driver_id AND tgt.race_id = upd.race_id)
WHEN MATCHED THEN 
UPDATE SET 
tgt.position = upd.position,
tgt.points = upd.points,
tgt.calculated_points = upd.calculated_points,
tgt.updated_date = current_timestamp
WHEN NOT MATCHED
THEN INSERT (race_year, team_name, driver_id, driver_name, race_id, position, points, calculated_points, created_date)
VALUES (race_year, team_name, driver_id, driver_name, race_id, position, points, calculated_points, current_timestamp)""")

In [0]:
%sql
SELECT * FROM f1_presentation.calculated_race_results